# NB01 — Data Preparation

Reuses `enigma_stress_phenotype_ml/data/labeled_pd.parquet` (215,051 proteins,
60 ENIGMA organisms). No Spark query required.

**Union binary label**: `essential_union = 1` if any `{stressor}_fit` column < −2.0.

**Genus mapping**: ENIGMA strain codes use abbreviations (e.g. "Keio", "pseudo1_N1B4"),
not binomial names. `str.split().str[0]` returns the full code for all organisms (no spaces),
making every organism its own unique "genus" — equivalent to LOOO. This notebook builds
a manual genus mapping using known ENIGMA strain identities, grouping organisms at the
correct phylogenetic level for true LOGO.

Multi-organism genera: Pseudomonas (12 strains), Ralstonia (4), Dickeya (3), Rhodanobacter (3),
Bacteroides (2), Burkholderia (2), Methanococcus (2). Remaining 34 organisms are singleton genera.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import GroupShuffleSplit

PROJ_ROOT  = Path.cwd().parent
DATA_DIR   = PROJ_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

SRC_DATA = PROJ_ROOT.parent / 'enigma_stress_phenotype_ml' / 'data'
FITNESS_THRESHOLD = -2.0

df = pd.read_parquet(SRC_DATA / 'labeled_pd.parquet')
print(f"Loaded labeled_pd.parquet: {df.shape}")
print(f"Unique organisms: {df['organism'].nunique()}")

Loaded labeled_pd.parquet: (215051, 96)
Unique organisms: 60


In [2]:
# Manual genus mapping — required because ENIGMA strain codes contain no spaces,
# so str.split().str[0] returns the full code and creates LOOO, not LOGO.
# Sources: FitnessBrowser metadata, ENIGMA consortium publications.
GENUS_MAP = {
    # Pseudomonas (12 organisms — largest multi-strain genus in the dataset)
    "Putida":                   "Pseudomonas",    # P. putida
    "pseudo1_N1B4":             "Pseudomonas",    # P. fluorescens ENIGMA isolate
    "pseudo3_N2E3":             "Pseudomonas",    # P. fluorescens ENIGMA isolate
    "pseudo5_N2C3_1":           "Pseudomonas",    # P. fluorescens ENIGMA isolate
    "pseudo6_N2E2":             "Pseudomonas",    # P. fluorescens ENIGMA isolate
    "pseudo13_GW456_L13":       "Pseudomonas",    # P. fluorescens ENIGMA isolate
    "SyringaeB728a":            "Pseudomonas",    # P. syringae pv. tomato B728a
    "SyringaeB728a_mexBdelta":  "Pseudomonas",    # P. syringae B728a (mexB KO mutant)
    "PS":                       "Pseudomonas",    # Pseudomonas sp.
    "PV4":                      "Pseudomonas",    # Pseudomonas sp. PV4
    "psRCH2":                   "Pseudomonas",    # Pseudomonas sp. RCH2
    "WCS417":                   "Pseudomonas",    # Pseudomonas sp. WCS417r
    # Ralstonia (4 organisms)
    "RalstoniaBSBF1503":        "Ralstonia",
    "RalstoniaGMI1000":         "Ralstonia",
    "RalstoniaPSI07":           "Ralstonia",
    "RalstoniaUW163":           "Ralstonia",
    # Dickeya (3 organisms)
    "Dda3937":                  "Dickeya",        # D. dadantii 3937
    "Ddia6719":                 "Dickeya",        # D. dianthicola
    "DdiaME23":                 "Dickeya",        # D. dianthicola ME23
    # Rhodanobacter (3 organisms)
    "rhodanobacter_10B01":      "Rhodanobacter",
    "rhodanobacter_R12":        "Rhodanobacter",
    "rhodanobacter_T8":         "Rhodanobacter",
    # Bacteroides / Phocaeicola (2 organisms)
    "Btheta":                   "Bacteroides",    # B. thetaiotaomicron
    "Bvulgatus_CL09T03C04":     "Bacteroides",    # B. vulgatus (now Phocaeicola)
    # Burkholderia (2 organisms)
    "Burk376":                  "Burkholderia",
    "Burkholderia_OAS925":      "Burkholderia",
    # Methanococcus (2 organisms)
    "Methanococcus_JJ":         "Methanococcus",  # M. jannaschii
    "Methanococcus_S2":         "Methanococcus",  # M. maripaludis S2
    # Singletons — known identity
    "BFirm":                    "Unknown_BFirm",  # ENIGMA firmicutes isolate (uncharacterized)
    "Bifido":                   "Bifidobacterium",
    "Brev2":                    "Brevundimonas",
    "Caulo":                    "Caulobacter",    # C. crescentus
    "CL21":                     "Unknown_CL21",
    "Cup4G11":                  "Cupriavidus",
    "Dino":                     "Deinococcus",
    "DvH":                      "Desulfovibrio",  # D. vulgaris Hildenborough
    "Dyella79":                 "Dyella",
    "HerbieS":                  "Herbaspirillum",
    "Kang":                     "Kangiella",
    "Keio":                     "Escherichia",    # E. coli K-12 (Keio collection)
    "Korea":                    "Unknown_Korea",
    "Koxy":                     "Klebsiella",     # K. oxytoca
    "Lysobacter_OAE881":        "Lysobacter",
    "Magneto":                  "Magnetospirillum",
    "Marino":                   "Marinobacter",
    "Miya":                     "Unknown_Miya",
    "MR1":                      "Shewanella",     # S. oneidensis MR-1
    "Mucilaginibacter_YX36":    "Mucilaginibacter",
    "MycoTube":                 "Mycobacterium",
    "Pedo557":                  "Pedobacter",
    "Phaeo":                    "Phaeobacter",
    "Ponti":                    "Pontibacter",
    "RPal_CGA009":              "Rhodopseudomonas", # R. palustris CGA009
    "SB2B":                     "Unknown_SB2B",
    "Smeli":                    "Sinorhizobium",  # S. meliloti
    "SynE":                     "Synechococcus",
    "Variovorax_OAS795":        "Variovorax",
    "Xantho":                   "Xanthomonas",
    "acidovorax_3H11":          "Acidovorax",
    "azobra":                   "Azospirillum",   # A. brasilense
}

all_orgs = df["organism"].unique()
unmapped = [o for o in all_orgs if o not in GENUS_MAP]
assert len(unmapped) == 0, f"Missing genus mappings: {unmapped}"
print(f"All {len(all_orgs)} organisms mapped")

# Multi-organism genera
from collections import Counter
genus_counts = Counter(GENUS_MAP.values())
multi = {g: c for g,c in genus_counts.items() if c > 1}
print(f"Multi-organism genera ({len(multi)}):")
for g, c in sorted(multi.items(), key=lambda x: -x[1]):
    orgs = [o for o,v in GENUS_MAP.items() if v == g]
    print(f"  {g}: {c} organisms → {', '.join(orgs)}")
print(f"Singleton genera: {sum(1 for c in genus_counts.values() if c == 1)}")

All 60 organisms mapped
Multi-organism genera (7):
  Pseudomonas: 12 organisms → Putida, pseudo1_N1B4, pseudo3_N2E3, pseudo5_N2C3_1, pseudo6_N2E2, pseudo13_GW456_L13, SyringaeB728a, SyringaeB728a_mexBdelta, PS, PV4, psRCH2, WCS417
  Ralstonia: 4 organisms → RalstoniaBSBF1503, RalstoniaGMI1000, RalstoniaPSI07, RalstoniaUW163
  Dickeya: 3 organisms → Dda3937, Ddia6719, DdiaME23
  Rhodanobacter: 3 organisms → rhodanobacter_10B01, rhodanobacter_R12, rhodanobacter_T8
  Bacteroides: 2 organisms → Btheta, Bvulgatus_CL09T03C04
  Burkholderia: 2 organisms → Burk376, Burkholderia_OAS925
  Methanococcus: 2 organisms → Methanococcus_JJ, Methanococcus_S2
Singleton genera: 32


In [3]:
# Fitness score columns (continuous)
fit_cols = [c for c in df.columns if c.endswith('_fit')]
print(f"Fitness columns ({len(fit_cols)})")

# Union essentiality: essential in ANY condition where fitness was measured
df['min_fitness'] = df[fit_cols].min(axis=1)
df['essential_union'] = (df['min_fitness'] < FITNESS_THRESHOLD).astype(int)

# Row-position protein_id (for feature file alignment in NB02)
df['protein_id'] = range(len(df))

# Apply manual genus mapping
df['genus'] = df['organism'].map(GENUS_MAP)

n_pos   = df['essential_union'].sum()
n_total = len(df)
print(f"\nUnion label (min fitness < {FITNESS_THRESHOLD}):")
print(f"  Essential:     {n_pos:,} ({100*n_pos/n_total:.2f}%)")
print(f"  Non-essential: {n_total-n_pos:,} ({100*(n_total-n_pos)/n_total:.2f}%)")
print()
print("Per-stressor positive rates:")
for col in fit_cols:
    n_tested = df[col].notna().sum()
    if n_tested == 0: continue
    n_ess = (df[col] < FITNESS_THRESHOLD).sum()
    print(f"  {col.replace('_fit',''):24s}: {n_ess:5d}/{n_tested:6d} ({100*n_ess/n_tested:.2f}%)")

Fitness columns (44)

Union label (min fitness < -2.0):
  Essential:     27,061 (12.58%)
  Non-essential: 187,990 (87.42%)

Per-stressor positive rates:
  Zn                      :  7300/142269 (5.13%)
  Cu                      :  8101/148953 (5.44%)
  Cd                      :   162/  6184 (2.62%)
  Co                      :  9229/150081 (6.15%)
  Ni                      :  8811/152937 (5.76%)
  Cr                      :  1054/ 35693 (2.95%)
  Hg                      :  1031/ 29263 (3.52%)
  Mn                      :   388/ 14370 (2.70%)
  Fe                      :  4637/115488 (4.02%)
  Se                      :   848/ 19702 (4.30%)
  Al                      :  6954/142561 (4.88%)
  Ampicillin              :  1334/ 61399 (2.17%)
  Gentamicin              :  4725/136303 (3.47%)
  Rifampicin              :  3939/102456 (3.84%)
  Chloramphenicol         :  7759/135706 (5.72%)
  Tetracycline            :  7397/140750 (5.26%)
  Phosphomycin            :  4580/116831 (3.92%)
  Ceftazidime 

In [4]:
# Organism-stratified split (no organism in both train and test)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, df['essential_union'], groups=df['organism']))

df_train = df.iloc[train_idx].reset_index(drop=True)
df_test  = df.iloc[test_idx].reset_index(drop=True)

overlap = set(df_train['organism'].unique()) & set(df_test['organism'].unique())
assert len(overlap) == 0, f"Organism leakage: {overlap}"

print("=== ORGANISM-STRATIFIED SPLIT ===")
print(f"Train: {len(df_train):,} proteins | {df_train['organism'].nunique()} organisms | {df_train['genus'].nunique()} genera")
print(f"Test:  {len(df_test):,} proteins | {df_test['organism'].nunique()} organisms | {df_test['genus'].nunique()} genera")
print(f"Organism overlap: 0 ✓")
print()
print(f"Train essential: {df_train['essential_union'].sum():,} ({100*df_train['essential_union'].mean():.2f}%)")
print(f"Test  essential: {df_test['essential_union'].sum():,} ({100*df_test['essential_union'].mean():.2f}%)")

# Genus LOGO preview
genus_summary = (
    df_train.groupby('genus')
    .agg(n_orgs=('organism','nunique'), n_proteins=('protein_id','count'),
         n_essential=('essential_union','sum'))
    .reset_index().sort_values('n_orgs', ascending=False)
)
genus_summary['pos_rate'] = genus_summary['n_essential'] / genus_summary['n_proteins']
print()
print(f"Training genera: {len(genus_summary)}")
print(f"Genera with ≥5 essential proteins (LOGO-eligible): {(genus_summary['n_essential']>=5).sum()}")
print()
print("Top genera by organism count:")
print(genus_summary.head(20).to_string(index=False))

=== ORGANISM-STRATIFIED SPLIT ===
Train: 166,705 proteins | 48 organisms | 32 genera
Test:  48,346 proteins | 12 organisms | 11 genera
Organism overlap: 0 ✓

Train essential: 22,406 (13.44%)
Test  essential: 4,655 (9.63%)

Training genera: 32
Genera with ≥5 essential proteins (LOGO-eligible): 27

Top genera by organism count:
           genus  n_orgs  n_proteins  n_essential  pos_rate
     Pseudomonas      10       39293         7722  0.196524
       Ralstonia       4       14995          401  0.026742
     Bacteroides       2        7262          941  0.129579
         Dickeya       2        7010          364  0.051926
   Rhodanobacter       2        5516            0  0.000000
   Methanococcus       2        2573          934  0.363000
 Bifidobacterium       1        1599           30  0.018762
      Acidovorax       1        3623          733  0.202319
   Desulfovibrio       1        2716          486  0.178940
          Dyella       1        3658            0  0.000000
     Escheri

In [5]:
# Save
meta_cols = ['protein_id', 'organism', 'genus', 'essential_union', 'min_fitness']
df[meta_cols].to_parquet(DATA_DIR / 'labeled_union.parquet', index=False)
df_train[meta_cols].to_parquet(DATA_DIR / 'labeled_train.parquet', index=False)
df_test[meta_cols].to_parquet(DATA_DIR / 'labeled_test.parquet', index=False)

json.dump({
    'train_protein_ids': df_train['protein_id'].tolist(),
    'test_protein_ids':  df_test['protein_id'].tolist(),
    'genus_map': GENUS_MAP,
}, open(DATA_DIR / 'split_protein_ids.json', 'w'))

print(f"Saved labeled_union.parquet  ({len(df):,} rows)")
print(f"Saved labeled_train.parquet  ({len(df_train):,} rows)")
print(f"Saved labeled_test.parquet   ({len(df_test):,} rows)")
print(f"Saved split_protein_ids.json")

Saved labeled_union.parquet  (215,051 rows)
Saved labeled_train.parquet  (166,705 rows)
Saved labeled_test.parquet   (48,346 rows)
Saved split_protein_ids.json
